# import libraries

In [2]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   


import sys
sys.path.append('../')
import helper_functions as hf

# generate recommendations

In [3]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 1
Very Important: Please Confirm the Iteration Number is Iteration 1
Very Important: Please Confirm the Iteration Number is Iteration 1


In [ ]:
df_design, ax_client = hf.run_optimizer(current_iteration=n, n_trials=6)

[INFO 06-03 10:50:38] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-03 10:50:38] ax.modelbridge.transforms.standardize_y: Outcome complexity is constant, within tolerance.
c:\Users\yunhe\anaconda3\envs\drug_surfactant\Lib\site-packages\botorch\models\utils\assorted.py:267: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([0.], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)


# process results

In [5]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

[INFO 06-03 01:18:14] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'s1': 47, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 's9': 85, 'surfactant_conc': 77, 'drug_conc': 89})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'s1': 92, 's2': 28, 's3': 51, 's4': 69, 's5': 27, 's6': 83, 's7': 70, 's8': 58, 's9': 36, 'surfactant_conc': 30, 'drug_conc': 3})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'s1': 53, 's2': 81, 's3': 8, 's4': 21, 's5': 17, 's6': 73, 's7': 48, 's8': 97, 's9': 20, 'surfactant_conc': 14, 'drug_conc': 29})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', parameters={'s1': 7, 's2': 7, 's3': 92, 's4': 84, 's5': 61, 's6': 49, 's7': 82, 's8': 24, 's9': 69, 'surfactant_conc': 68, 'drug_conc': 69})),
 4: Trial(

In [6]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [7]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: B1
Deep plate will start at: B1

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [8]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_1.py


In [4]:
df_absorbance = hf.process_absorbance(iteration=n)

In [7]:
results = hf.build_results(n, df_conc, df_absorbance)

In [8]:
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,s9,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,6,0,0,100,100,57,100,68,100,100,0.5,19.50,0,0.000,7
1,7,0,0,0,100,0,100,100,100,100,0.5,0.25,1,0.025,5
2,8,0,0,100,100,59,100,58,0,0,0.5,25.00,0,0.000,5
3,9,0,0,100,0,0,0,100,100,100,0.5,0.25,1,0.025,4
4,10,0,100,100,100,0,100,100,100,100,0.5,25.00,0,0.000,7
5,11,0,0,0,0,61,100,100,100,0,0.5,25.00,0,0.000,4


In [9]:
norm_results = hf.normalize_data(results, 'normalize')

In [10]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,s9,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,6,0,0,100,100,57,100,68,100,100,0.01,19.50,0,0.00,0.777778
1,7,0,0,0,100,0,100,100,100,100,0.01,0.25,1,0.01,0.555556
2,8,0,0,100,100,59,100,58,0,0,0.01,25.00,0,0.00,0.555556
3,9,0,0,100,0,0,0,100,100,100,0.01,0.25,1,0.01,0.444444
4,10,0,100,100,100,0,100,100,100,100,0.01,25.00,0,0.00,0.777778
5,11,0,0,0,0,61,100,100,100,0,0.01,25.00,0,0.00,0.444444


# load the results to the optimizer

In [11]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-03 10:53:40] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-03 10:53:40] ax.service.ax_client: Completed trial 6 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.777778, None)}.
[INFO 06-03 10:53:40] ax.service.ax_client: Completed trial 7 with data: {'micelle_drug_conc': (0.01, None), 'surfactant_conc': (0.01, None), 'complexity': (0.555556, None)}.
[INFO 06-03 10:53:40] ax.service.ax_client: Completed trial 8 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.555556, None)}.
[INFO 06-03 10:53:40] ax.service.ax_client: Completed trial 9 with data: {'micelle_drug_conc': (0.01, None), 'surfactant_conc': (0.01, None), 'complexity': (0.444444, None)}.
[INFO 06-03 10:53:40] ax.service.ax_client: Completed trial 10 with data: {'mic

AxClient(experiment=Experiment(drug_surfactant))